In [1]:
import sys, os, ast, time, platform, random
import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath('..'))
from src.models import MLP
from src.train import treinar

# 1. Especificações do ambiente (obrigatório no relatório)
env_info = {
    "SO": f"{platform.system()} {platform.release()}",
    "Processador": platform.processor(),
    "Nucleos_fisicos": psutil.cpu_count(logical=False),
    "Nucleos_logicos": psutil.cpu_count(logical=True),
    "RAM_total_GB": round(psutil.virtual_memory().total / (1024**3), 2),
    "Python": platform.python_version(),
    "PyTorch": torch.__version__,
    "Torch_num_threads": torch.get_num_threads(),
    "CUDA_disponivel": torch.cuda.is_available(),
}
df_env = pd.DataFrame([env_info])
df_env.to_csv('../reports/tables/ambiente_computacional.csv', index=False)
display(df_env.T.rename(columns={0: "Valor"}))

# Constantes para estimativa de Energia e CO2 (Brasil: 45,8 gCO2/kWh)
POTENCIA_CPU_WATTS = 45.0  # TDP médio típico de CPU de notebook/desktop em carga
FATOR_CO2_BR = 45.8        # gCO2 / kWh (conforme aula e PDF)

def estimar_energia_co2(tempo_s, potencia_w=POTENCIA_CPU_WATTS):
    kwh = (potencia_w * (tempo_s / 3600.0)) / 1000.0
    co2_g = kwh * FATOR_CO2_BR
    return kwh, co2_g

def calcular_macs_mlp(modelo):
    """Para camadas nn.Linear, MACs por amostra = in_features * out_features."""
    macs = 0
    n_camadas = 0
    for m in modelo.modules():
        if isinstance(m, nn.Linear):
            macs += m.in_features * m.out_features
            n_camadas += 1
    return macs, n_camadas

def medir_inferencia(modelo, X, n_rep=30, n_aquecimento=5):
    modelo.eval()
    proc = psutil.Process()
    with torch.inference_mode():
        for _ in range(n_aquecimento):
            modelo(X)
        mem_antes = proc.memory_info().rss
        tempos = []
        for _ in range(n_rep):
            inicio = time.perf_counter()
            modelo(X)
            tempos.append(time.perf_counter() - inicio)
        mem_depois = proc.memory_info().rss
    media, desvio = float(np.mean(tempos)), float(np.std(tempos))
    return {
        "tempo_medio_s": media,
        "tempo_desvio_s": desvio,
        "tempo_por_amostra_ms": 1000.0 * media / len(X),
        "vazao_amostras_s": len(X) / media,
        "delta_memoria_kb": (mem_depois - mem_antes) / 1024.0,
        "ram_residente_mb": mem_depois / (1024**2),
    }

def criar_otimizador(nome, params, lr, wd):
    if nome == "SGD": return torch.optim.SGD(params, lr=lr, weight_decay=wd)
    if nome == "SGD_momentum": return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd)
    if nome == "RMSprop": return torch.optim.RMSprop(params, lr=lr, weight_decay=wd)
    if nome == "Adam": return torch.optim.Adam(params, lr=lr, weight_decay=wd)
    if nome == "AdamW": return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    raise ValueError(nome)

def carregar_campeao(caminho_csv):
    row = pd.read_csv(caminho_csv).iloc[0]
    return {
        "hidden": ast.literal_eval(row["hidden"]) if isinstance(row["hidden"], str) else row["hidden"],
        "activation": row["activation"], "opt": row["opt"],
        "lr": float(row["lr"]), "batch_size": int(row["batch_size"]),
        "dropout": float(row["dropout"]), "weight_decay": float(row["weight_decay"]),
    }

CFG_REG = carregar_campeao('../reports/tables/busca_final_regressao.csv')
CFG_CLF = carregar_campeao('../reports/tables/busca_final_classificacao.csv')

,Valor
SO,Windows 11
Processador,"Intel64 Family 6 Model 140 Stepping 1, Genuine..."
Nucleos_fisicos,2
Nucleos_logicos,4
RAM_total_GB,3.73
Python,3.13.5
PyTorch,2.15.0+cpu
Torch_num_threads,2
CUDA_disponivel,False


In [2]:
def profile_treino(tarefa, cfg):
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    proc = psutil.Process()

    if tarefa == "regressao":
        df = pd.read_csv('../data/processed/train_regression.csv')
        X = df.drop(columns=['Mental_Wellbeing_Score']).values
        y = df['Mental_Wellbeing_Score'].values
        X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42)
        sc_x = StandardScaler().fit(X_tr)
        sc_y = StandardScaler().fit(y_tr.reshape(-1, 1))
        dl_tr = DataLoader(TensorDataset(torch.tensor(sc_x.transform(X_tr), dtype=torch.float32),
                                         torch.tensor(sc_y.transform(y_tr.reshape(-1, 1)), dtype=torch.float32)),
                           batch_size=cfg["batch_size"], shuffle=True)
        dl_va = DataLoader(TensorDataset(torch.tensor(sc_x.transform(X_va), dtype=torch.float32),
                                         torch.tensor(sc_y.transform(y_va.reshape(-1, 1)), dtype=torch.float32)),
                           batch_size=64, shuffle=False)
        out_dim, loss_fn = 1, nn.MSELoss()
    else:
        df = pd.read_csv('../data/processed/train_classification.csv')
        X = df.drop(columns=['Addiction_Level']).values
        y = df['Addiction_Level'].values
        X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        sc_x = StandardScaler().fit(X_tr)
        dl_tr = DataLoader(TensorDataset(torch.tensor(sc_x.transform(X_tr), dtype=torch.float32),
                                         torch.tensor(y_tr, dtype=torch.long)),
                           batch_size=cfg["batch_size"], shuffle=True)
        dl_va = DataLoader(TensorDataset(torch.tensor(sc_x.transform(X_va), dtype=torch.float32),
                                         torch.tensor(y_va, dtype=torch.long)),
                           batch_size=64, shuffle=False)
        out_dim, loss_fn = 4, nn.CrossEntropyLoss()

    modelo = MLP(X_tr.shape[1], out_dim, cfg["hidden"], cfg["activation"], cfg["dropout"])
    otim = criar_otimizador(cfg["opt"], modelo.parameters(), cfg["lr"], cfg["weight_decay"])

    ram_antes = proc.memory_info().rss / (1024**2)
    t0 = time.perf_counter()
    hist = treinar(modelo, dl_tr, dl_va, loss_fn, otim, n_epocas=200, paciencia=20)
    tempo_s = time.perf_counter() - t0
    ram_depois = proc.memory_info().rss / (1024**2)

    kwh, co2_g = estimar_energia_co2(tempo_s)
    return {
        "Tarefa": tarefa,
        "Epocas_executadas": len(hist["val"]),
        "Tempo_treino_s": round(tempo_s, 4),
        "RAM_treino_MB": round(max(ram_antes, ram_depois), 2),
        "Energia_treino_kWh": kwh,
        "CO2_treino_g": co2_g,
    }

df_prof_treino = pd.DataFrame([
    profile_treino("regressao", CFG_REG),
    profile_treino("classificacao", CFG_CLF)
])
df_prof_treino.to_csv('../reports/tables/profiling_treino.csv', index=False)
display(df_prof_treino)

,Tarefa,Epocas_executadas,Tempo_treino_s,RAM_treino_MB,Energia_treino_kWh,CO2_treino_g
0,regressao,39,2.1580,312.77,0.000027,0.001235
1,classificacao,23,1.5358,304.44,0.000019,0.000879


In [3]:
rows_inf = []

for tarefa, ckpt_path, test_csv, target_col in [
    ("regressao", "../models/modelo_regressao.pt", "../data/processed/test_regression.csv", "Mental_Wellbeing_Score"),
    ("classificacao", "../models/modelo_classificacao.pt", "../data/processed/test_classification.csv", "Addiction_Level"),
]:
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    modelo = MLP(**ckpt["config"])
    modelo.load_state_dict(ckpt["state_dict"])
    modelo.eval()

    df_te = pd.read_csv(test_csv)
    X_te = torch.tensor(df_te[ckpt["features"]].values, dtype=torch.float32)
    X_te = (X_te - ckpt["x_media"]) / ckpt["x_desvio"]

    n_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    macs, n_camadas = calcular_macs_mlp(modelo)
    tam_disco_kb = os.path.getsize(ckpt_path) / 1024.0
    tam_teorico_kb = (n_params * 4) / 1024.0

    # Inferência em lote inteiro vs. 1 amostra
    res_lote = medir_inferencia(modelo, X_te, n_rep=30, n_aquecimento=5)
    res_unit = medir_inferencia(modelo, X_te[:1], n_rep=30, n_aquecimento=5)
    kwh_inf, co2_inf = estimar_energia_co2(res_lote["tempo_medio_s"])

    rows_inf.append({
        "Tarefa": tarefa,
        "Parametros": n_params,
        "Camadas_Lineares": n_camadas,
        "MACs_por_amostra": macs,
        "Tamanho_disco_KB": round(tam_disco_kb, 2),
        "Tamanho_teorico_float32_KB": round(tam_teorico_kb, 2),
        "Lote_tempo_medio_ms": round(res_lote["tempo_medio_s"] * 1000, 4),
        "Lote_tempo_std_ms": round(res_lote["tempo_desvio_s"] * 1000, 4),
        "Lote_ms_por_amostra": round(res_lote["tempo_por_amostra_ms"], 5),
        "Lote_vazao_amostras_s": round(res_lote["vazao_amostras_s"], 1),
        "Unit_tempo_medio_ms": round(res_unit["tempo_medio_s"] * 1000, 4),
        "Unit_vazao_amostras_s": round(res_unit["vazao_amostras_s"], 1),
        "RAM_inf_MB": round(res_lote["ram_residente_mb"], 2),
        "Energia_inf_lote_kWh": kwh_inf,
        "CO2_inf_lote_g": co2_inf,
    })

df_prof_inf = pd.DataFrame(rows_inf)
df_prof_inf.to_csv('../reports/tables/profiling_inferencia.csv', index=False)
display(df_prof_inf.T)

,0,1
Tarefa,regressao,classificacao
Parametros,2049,6340
Camadas_Lineares,3,3
MACs_por_amostra,1984,6208
Tamanho_disco_KB,12.7,29.69
Tamanho_teorico_float32_KB,8.0,24.77
Lote_tempo_medio_ms,0.1838,0.1002
Lote_tempo_std_ms,0.035,0.0023
Lote_ms_por_amostra,0.00061,0.00033
Lote_vazao_amostras_s,1632653.1,2994510.1


In [4]:
ev_reg = pd.read_csv('../reports/tables/avaliacao_regressao.csv')
ev_clf = pd.read_csv('../reports/tables/avaliacao_classificacao.csv')
cv_df = pd.read_csv('../reports/tables/kfold_5_final.csv')

resumo = [
    {
        "Tarefa": "Regressão",
        "Config_Final": str(CFG_REG),
        "Metrica_Principal": "RMSE",
        "Treino": round(ev_reg.query("Modelo == 'Final (Recarregado)' and Conjunto == 'Treino'")["RMSE"].values[0], 4),
        "Teste": round(ev_reg.query("Modelo == 'Final (Recarregado)' and Conjunto == 'Teste'")["RMSE"].values[0], 4),
        "KFold5_Media": round(cv_df.query("Tarefa == 'Regressão'")["Media_5Fold"].values[0], 4),
        "KFold5_Std": round(cv_df.query("Tarefa == 'Regressão'")["Std_5Fold"].values[0], 4),
        "Parametros": df_prof_inf.query("Tarefa == 'regressao'")["Parametros"].values[0],
        "Disco_KB": df_prof_inf.query("Tarefa == 'regressao'")["Tamanho_disco_KB"].values[0],
        "Treino_s": df_prof_treino.query("Tarefa == 'regressao'")["Tempo_treino_s"].values[0],
        "Vazao_Lote_amostras_s": df_prof_inf.query("Tarefa == 'regressao'")["Lote_vazao_amostras_s"].values[0],
    },
    {
        "Tarefa": "Classificação",
        "Config_Final": str(CFG_CLF),
        "Metrica_Principal": "F1_macro",
        "Treino": round(ev_clf.query("Modelo == 'Final (Recarregado)' and Conjunto == 'Treino'")["F1_macro"].values[0], 4),
        "Teste": round(ev_clf.query("Modelo == 'Final (Recarregado)' and Conjunto == 'Teste'")["F1_macro"].values[0], 4),
        "KFold5_Media": round(cv_df.query("Tarefa == 'Classificação'")["Media_5Fold"].values[0], 4),
        "KFold5_Std": round(cv_df.query("Tarefa == 'Classificação'")["Std_5Fold"].values[0], 4),
        "Parametros": df_prof_inf.query("Tarefa == 'classificacao'")["Parametros"].values[0],
        "Disco_KB": df_prof_inf.query("Tarefa == 'classificacao'")["Tamanho_disco_KB"].values[0],
        "Treino_s": df_prof_treino.query("Tarefa == 'classificacao'")["Tempo_treino_s"].values[0],
        "Vazao_Lote_amostras_s": df_prof_inf.query("Tarefa == 'classificacao'")["Lote_vazao_amostras_s"].values[0],
    }
]

df_resumo = pd.DataFrame(resumo)
df_resumo.to_csv('../reports/tables/resumo_final.csv', index=False)
display(df_resumo)

# Teste final do script independente com os modelos campeões atualizados
print("\n--- Validação final via src/inference.py ---")
!python ../src/inference.py --modelo ../models/modelo_regressao.pt --entrada ../data/processed/test_regression.csv
print("-" * 50)
!python ../src/inference.py --modelo ../models/modelo_classificacao.pt --entrada ../data/processed/test_classification.csv

,Tarefa,Config_Final,Metrica_Principal,Treino,Teste,KFold5_Media,KFold5_Std,Parametros,Disco_KB,Treino_s,Vazao_Lote_amostras_s
0,Regressão,"{'hidden': [32, 32], 'activation': 'GELU', 'op...",RMSE,0.3299,0.5142,0.4673,0.0124,2049,12.70,2.1580,1632653.1
1,Classificação,"{'hidden': [64, 64], 'activation': 'Tanh', 'op...",F1_macro,0.8112,0.7761,0.7579,0.0258,6340,29.69,1.5358,2994510.1



--- Validação final via src/inference.py ---
Inferência concluída para 300 amostras.
Primeiras 10 previsões: [-0.5210062  -0.5343601   0.08531758 -0.5506751  -1.1658249   0.45874643
  1.1369058  -1.6794952  -1.2994591  -1.2054751 ]
--------------------------------------------------
Inferência concluída para 300 amostras.
Primeiras 10 previsões: [1 2 2 0 3 0 3 0 1 3]


In [7]:
df_hp = pd.read_csv('../logs/hiperparametros.csv')

for tarefa in df_hp['Tarefa'].unique():
    sub = df_hp[df_hp['Tarefa'] == tarefa]
    is_reg = "reg" in str(tarefa).lower()
    
    linhas = []
    for hp, grp in sub.groupby('Hiperparametro'):
        melhor = grp['Metrica_val_media'].min() if is_reg else grp['Metrica_val_media'].max()
        pior = grp['Metrica_val_media'].max() if is_reg else grp['Metrica_val_media'].min()
        impacto = abs(pior - melhor)
        std_medio = grp['Metrica_val_std'].mean()
        linhas.append({
            "Hiperparametro": hp,
            "Melhor_Val": round(melhor, 4),
            "Pior_Val": round(pior, 4),
            "Impacto_Delta": round(impacto, 4),
            "Std_Medio_Sementes": round(std_medio, 4),
            "Significativo": "Sim" if impacto > std_medio else "Desprezível (<= std)"
        })
        
    df_sens = pd.DataFrame(linhas).sort_values("Impacto_Delta", ascending=False)
    nome_arq = f'../reports/tables/tabela_sensibilidade_{str(tarefa).lower()}.csv'
    df_sens.to_csv(nome_arq, index=False)
    print(f"\n=== Sensibilidade: {tarefa} ===")
    display(df_sens)


=== Sensibilidade: regressao ===


,Hiperparametro,Melhor_Val,Pior_Val,Impacto_Delta,Std_Medio_Sementes,Significativo
4,Learning_Rate,0.4891,0.5687,0.0796,0.0149,Sim
5,Neuronios,0.4884,0.5670,0.0787,0.0233,Sim
3,Dropout,0.4795,0.5577,0.0782,0.0057,Sim
6,Otimizador,0.5116,0.5769,0.0653,0.0156,Sim
7,Weight_Decay,0.4932,0.5577,0.0646,0.0163,Sim
2,Camadas,0.5139,0.5769,0.0630,0.0115,Sim
0,Ativacao,0.5152,0.5685,0.0533,0.0135,Sim
1,Batch_Size,0.5454,0.5725,0.0271,0.0122,Sim



=== Sensibilidade: classificacao ===


,Hiperparametro,Melhor_Val,Pior_Val,Impacto_Delta,Std_Medio_Sementes,Significativo
6,Otimizador,0.7404,0.4426,0.2978,0.0224,Sim
4,Learning_Rate,0.7324,0.7017,0.0308,0.0123,Sim
1,Batch_Size,0.7333,0.7066,0.0267,0.0122,Sim
0,Ativacao,0.7527,0.7280,0.0247,0.0076,Sim
7,Weight_Decay,0.7488,0.7245,0.0242,0.0091,Sim
5,Neuronios,0.7433,0.7243,0.0190,0.0116,Sim
3,Dropout,0.7412,0.7253,0.0159,0.0090,Sim
2,Camadas,0.7324,0.7199,0.0125,0.0105,Sim
